# Complete Data Pipeline and Exploratory Data Analysis

## 1. Project Overview
In this comprehensive data pipeline mini-project, we will perform data ingestion, cleaning, exploratory data analysis (EDA), and feature engineering on the classic **Titanic Dataset**. Our ultimate goal is to prepare the dataset for a Machine Learning model that predicts passenger survival.

## 2. Import Libraries
We start by loading the necessary Python libraries for data manipulation, visualization, and machine learning preprocessing.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing & ML libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier

# Configure visualization settings
sns.set_theme(style="whitegrid")
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

## 3. Data Collection
We load the `train.csv` file from our local `Dataset` directory. This acts as our primary dataset for building the pipeline.

In [ ]:
# Load the dataset
data_path = 'Dataset/train.csv'
df = pd.read_csv(data_path)

# Display the first few rows
display(df.head())

## 4. Data Cleaning
Data cleaning is crucial. We will:
- Check for missing values and handle them appropriately.
- Drop rows/columns that contain mostly missing values or are not useful for ML.
- Remove duplicates.
- Fix incorrect data types.

In [ ]:
# 4.1 Duplicate Check
duplicates = df.duplicated().sum()
print(f"Number of duplicated rows: {duplicates}")
if duplicates > 0:
    df = df.drop_duplicates()

# 4.2 Missing Values Inspection
print("\nMissing values per column:")
print(df.isnull().sum())

# 4.3 Handling Missing Values
# Age: Fill with median age
df['Age'] = df['Age'].fillna(df['Age'].median())

# Cabin: Too many missing values, let's substitute with 'Unknown' or drop. We will replace with 'U'
df['Cabin'] = df['Cabin'].fillna('U')

# Embarked: Only 2 missing, fill with mode (most frequent)
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

print("\nMissing values after cleaning:")
print(df.isnull().sum())

### Detecting and Handling Outliers
We use the IQR (Interquartile Range) method to identify outliers in numerical columns like `Fare`.

In [ ]:
def handle_outliers(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Cap the outliers rather than dropping to preserve dataset size
    data[column] = np.where(data[column] > upper_bound, upper_bound, data[column])
    data[column] = np.where(data[column] < lower_bound, lower_bound, data[column])
    return data

# Apply outlier handling to Fare
df = handle_outliers(df, 'Fare')

## 5. Exploratory Data Analysis (EDA)
Here, we uncover trends, correlations, and general statistics.

In [ ]:
# Statistical summary for numerical columns
display(df.describe())

# Correlation Matrix to see relationships between variables
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
print("\nCorrelation of numerical features with Survived:")
print(df[numeric_cols].corr()['Survived'].sort_values(ascending=False))

## 6. Data Visualization (8 Required Plots)
Visualizing distributions and relationships provides deeper insights.

In [ ]:
plt.figure(figsize=(20, 16))

# 1. Histogram - Age Distribution
plt.subplot(3, 3, 1)
sns.histplot(df['Age'], kde=True, bins=30, color='skyblue')
plt.title('1. Histogram: Age Distribution')

# 2. Boxplot - Fare by Class
plt.subplot(3, 3, 2)
sns.boxplot(x='Pclass', y='Fare', data=df, palette='Set2')
plt.title('2. Boxplot: Fare by Passenger Class')

# 3. Heatmap - Correlation Matrix
plt.subplot(3, 3, 3)
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('3. Heatmap: Correlation Matrix')

# 4. Countplot - Survival Count by Sex
plt.subplot(3, 3, 4)
sns.countplot(x='Survived', hue='Sex', data=df, palette='pastel')
plt.title('4. Countplot: Survival by Sex')

# 5. Scatter Plot - Age vs Fare
plt.subplot(3, 3, 5)
sns.scatterplot(x='Age', y='Fare', hue='Survived', data=df, palette='rainbow', alpha=0.7)
plt.title('5. Scatter Plot: Age vs Fare')

# 6. Line Plot - Average Fare over Age groups
plt.subplot(3, 3, 6)
age_fare = df.groupby(pd.cut(df['Age'], bins=10))['Fare'].mean().reset_index()
age_fare['Age_mid'] = age_fare['Age'].apply(lambda x: x.mid)
sns.lineplot(x='Age_mid', y='Fare', data=age_fare, marker='o', color='purple')
plt.title('6. Line Plot: Avg Fare across Age Brackets')

# 7. Bar Chart - Survival Rate by Passenger Class
plt.subplot(3, 3, 7)
sns.barplot(x='Pclass', y='Survived', data=df, palette='viridis', ci=None)
plt.title('7. Bar Chart: Survival Rate by Ticket Class')

plt.tight_layout()
plt.show()

# 8. Pairplot - Overall Relationships
print("\n8. Pairplot: Numerical Features Relationships")
sns.pairplot(df[['Survived', 'Pclass', 'Age', 'Fare', 'SibSp']], hue='Survived', palette='husl', corner=True)
plt.show()

## 7. Feature Engineering
Enhancing the dataset makes learning easier for algorithms.
- Creating new meaningful features (`FamilySize`, `IsAlone`).
- Dropping irrelevant columns (`PassengerId`, `Name`, `Ticket`, `Cabin`).
- Encoding categorical fields.
- Standardizing Numerical values.

In [ ]:
# Create New Feature 1: FamilySize
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# Create New Feature 2: IsAlone (1 if traveling alone, 0 otherwise)
df['IsAlone'] = 1
df.loc[df['FamilySize'] > 1, 'IsAlone'] = 0

# Drop unneeded columns
drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin']
df_clean = df.drop(columns=drop_cols)

# Encode Categorical Variables (Sex and Embarked)
le = LabelEncoder()
df_clean['Sex'] = le.fit_transform(df_clean['Sex'])

# For 'Embarked', One-Hot Encoding is generally better for unordered categories
df_engine = pd.get_dummies(df_clean, columns=['Embarked'], drop_first=True)

display(df_engine.head())

## 8. Final Output & Bonus: Machine Learning Prep
We have a clean, numerical dataset. Let's do the train-test split, scale features, and suggest an ML model.

In [ ]:
# Separate Features (X) and Target (y)
X = df_engine.drop('Survived', axis=1)
y = df_engine['Survived']

# Feature Scaling (Standardization)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

# Summary of Insights for ML:
# - Random Forest Classifier or XGBoost are highly recommended for this type of tabular classification task.
# - They handle non-linear relationships well and inherently calculate feature importance.

# Simple Model Initialization (Bonus)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print(f"\nRandom Forest Accuracy on Training Data: {model.score(X_train, y_train):.2f}")
print(f"Random Forest Accuracy on Test Data: {model.score(X_test, y_test):.2f}")

## Final Summary of Insights
1. **Demographics:** The dataset skewed slightly younger, largely in their 20s and 30s.
2. **Survival by Sex:** Females had a significantly higher survival rate compared to males.
3. **Survival by Class:** Passengers in 1st class had better survival rates than those in 3rd class, indicating socioeconomic status played a role in rescue priority.
4. **Family Size:** Traveling with 1-3 family members marginally improved survival odds compared to traveling alone or in very large groups.
5. **Data Health:** The dataset is now clean, free of major outliers in critical variance features, and purely numeric—fully optimized for Machine Learning consumption.